Data set is in the supplementary of https://onlinelibrary.wiley.com/doi/abs/10.1111/biom.12537

Full article is available at https://pmc.ncbi.nlm.nih.gov/articles/PMC5654727/


From the supplementary data directly
```
TIME: time from treatment initiation (negative before treatment initiation)
- ID: patient’s identifiants
- Y = log(PSA+1) if YTYPE equals to 1
= vital status (1 if death, 0 if censored) if YTYPE equals to 2
- CENS = 1 if PSA is below the limit of quantification (0.1 ng.ml−1
) and 0 otherwise
- YTYPE: type of observation, 1 for PSA, 2 for vital status
- TIME ENDtr: covariate that indicates the day of end of treatment
```

The model parametrization is reworked slightly:
- PSA at baseline is considered a patient descriptor, extracted from the data
- PSA at baseline is used to infer the total amount of cells at baseline (S+R)
- A new descriptor is added: baseline ratio of S/R, to be estimated with SAEM
- Both types of cells have the same exact growth rate $`\alpha`$. The R_F descriptor is removed

In [ ]:
import pandas as pd
from plotnine import *
import numpy as np

from vpop_calibration.api import *

%load_ext autoreload
%autoreload 2

In [ ]:
df_raw = pd.read_csv("./data/JM400.txt", sep="\t")
print(df_raw.columns)

# quick visualization
(
    ggplot(df_raw, aes(x="TIME", y="Y", color="factor(ID)"))
    + geom_line()
    + theme(legend_position="none")
    + facet_wrap("YTYPE")
)

In [ ]:
# Remove vital status observation at t=0 (no added information)
df_processed = df_raw.loc[~((df_raw["YTYPE"] == 2) & (df_raw["TIME"] == 0))]

# Extract first observation time per patient
min_time_per_patient = (
    df_processed.groupby("ID")["TIME"].min().astype(float).rename("min_time")
)
# Shift all observed times by this value so that they start at 0
df_processed = df_processed.merge(min_time_per_patient, on="ID")
df_processed["time"] = df_processed["TIME"] - df_processed["min_time"]
df_processed["treatment__start"] = -df_processed["min_time"]
df_processed["treatment__end"] = df_processed["TIME_ENDtr"] - df_processed["min_time"]
df_processed = df_processed.rename(columns={"ID": "id", "Y": "value"}).drop(
    columns=["min_time", "TIME", "TIME_ENDtr"]
)
time_columns = ["time", "treatment__start", "treatment__end"]
df_processed[time_columns] = df_processed[time_columns].apply(
    lambda t: t * 24 * 60 * 60
)

# Extract survival data
surv_df = (
    df_processed.loc[(df_processed["YTYPE"] == 2) & (df_processed["time"] > 0)][
        ["id", "time", "value"]
    ]
    .rename(columns={"value": "event_status", "time": "event_time"})
    .drop_duplicates()
)
surv_df["event_status"] = surv_df["event_status"].apply(lambda status: status == 1)
surv_df["hazard_name"] = "_hazard"


# Extract longitudinal data
biomarkers_df = df_processed.loc[df_processed["YTYPE"] == 1][
    ["id", "time", "value", "treatment__start", "treatment__end"]
].drop_duplicates()
biomarkers_df["output_name"] = "log__psa"

# Extract baseline PSA
baseline_df = df_processed.loc[
    (df_processed["YTYPE"] == 1) & (df_processed["time"] == 0.0)
][["id", "value"]]
baseline_df["PSA__baseline"] = baseline_df["value"].apply(lambda psa: np.exp(psa) - 1)
baseline_df = baseline_df.drop(columns="value")

# Combine the two data sets
obs_df = biomarkers_df.merge(surv_df, on="id").merge(baseline_df, on="id")
display(obs_df)

patients = obs_df["id"].drop_duplicates()
patients_training = patients.sample(frac=0.8)

training_df = obs_df.loc[obs_df["id"].isin(patients_training)]

In [ ]:
model = StructuralSbml(
    model_path="./model/CM.xml",
    solving_options_path="./model/solving_options.yaml",
    inputs=[
        "Nmax",
        "alpha",
        "epsilon",
        "R__E",
        "PSA__baseline",
        "treatment__start",
        "treatment__end",
        "beta1__survival",
        "beta2__survival",
        "k__weibull",
        "lambda__weibull",
        "baseline__log__ratio__S__R",
    ],
    outputs=["log__psa", "log__hazard", "cumulative__hazard"],
)

In [ ]:
input_params = {
    "model_intrinsic": {
        "k__weibull": {"prior": 1.2},
        "lambda__weibull": {"prior": 800},
    },
    "pdu": {
        "Nmax": {"prior": 55, "prior_omega": 0.5},
        "alpha": {"prior": 0.06, "prior_omega": 0.5},
        "epsilon": {
            "prior": 0.4,
            "prior_omega": 0.5,
            "constraint": {"low": 0.0, "high": 1.0},
        },
        "R__E": {
            "prior": 0.8,
            "prior_omega": 0.5,
            "constraint": {"low": 0.0, "high": 1.0},
        },
        "baseline__log__ratio__S__R": {"prior": 2.0, "prior_omega": 0.5},
    },
    "error_model": {"log__psa": {"error_type": "additive", "sigma": 0.5}},
    "pdk": ["treatment__start", "treatment__end", "PSA__baseline"],
    "time_to_event": {
        "hazard_name": "_hazard",
        "coefficients": {
            "beta1__survival": {"prior": 0.0},
            "beta2__survival": {"prior": 0.0},
        },
    },
}

config = Config(
    saem=SaemConfigDict(
        optim_max_iter=5,
        nb_iter_burnin=0,
        nb_iter_smoothing=50,
        nb_iter_learning=50,
        plot_frames=1,
        plot_columns=5,
    )
)

nlme_model = NlmeModel(
    df=training_df, structural_model=model, input_params=input_params, config=config
)

In [ ]:
nlme_model.optimizer.run()

In [ ]:
nlme_model.diagnostics.sample_conditional_distribution(nb_samples=200)

In [ ]:
nlme_model.plot.map_estimates()

In [ ]:
nlme_model.plot.map_estimates_gof()

In [ ]:
nlme_model.plot.vpc()